# Runprep Convergence Analysis

This notebook reads `provenance.json` generated by `pynta/preprocessing/runprep.py` and plots:

- ecut convergence for each k-point mesh
- k-point convergence for each ecut value
- lattice constant energy scan if available

If `provenance.json` is not in the notebook root, the notebook will search parent directories.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
provenance_path = None
for candidate in candidates:
    p = candidate / 'provenance.json'
    if p.exists():
        provenance_path = p
        break

if provenance_path is None:
    raise FileNotFoundError('Unable to find provenance.json in the current or parent directories.')

print(f'Using provenance: {provenance_path}')
with open(provenance_path, 'r') as f:
    provenance = json.load(f)

steps = provenance.get('steps', [])
print(f'Found {len(steps)} provenance steps.')


In [ ]:
convergence_steps = [s for s in steps if s.get('step') == 'run_convergence']
lattice_steps = [s for s in steps if s.get('step') == 'run_lattice_constant_scan']

print(f'run_convergence entries: {len(convergence_steps)}')
print(f'run_lattice_constant_scan entries: {len(lattice_steps)}')


In [ ]:
if len(convergence_steps) == 0:
    raise RuntimeError('No run_convergence results available in provenance.json.')

for idx, step in enumerate(convergence_steps, start=1):
    results = step['outputs']['results']
    ecut_vals = sorted({r[0] for r in results})
    kpt_vals = sorted({tuple(r[1]) for r in results})

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for kpt in kpt_vals:
        energies = [next(r[2] for r in results if r[0] == ecut and tuple(r[1]) == kpt) for ecut in ecut_vals]
        axes[0].plot(ecut_vals, energies, marker='o', label=str(kpt))

    axes[0].set_title(f'Energy vs ecut (run_convergence #{idx})')
    axes[0].set_xlabel('ecut (Ry)')
    axes[0].set_ylabel('Total energy (eV)')

    for ecut in ecut_vals:
        kpts_sorted = sorted(kpt_vals, key=lambda k: (k[0], k[1], k[2]))
        energies = [next(r[2] for r in results if r[0] == ecut and tuple(r[1]) == kpt) for kpt in kpts_sorted]
        axes[1].plot([str(kpt) for kpt in kpts_sorted], energies, marker='o', label=f'ecut={ecut}')

    axes[1].set_title(f'Energy vs k-points (run_convergence #{idx})')
    axes[1].set_xlabel('k-point mesh')
    axes[1].set_ylabel('Total energy (eV)')

    plt.show()

    if len(convergence_steps) > 1:
        print(f'--- End of run_convergence #{idx} ---')

    if idx < len(convergence_steps):
        print()


In [ ]:
if len(lattice_steps) == 0:
    print('No lattice constant scan results found. Run run_lattice_constant_scan first to collect scan data.')
else:
    for idx, step in enumerate(lattice_steps, start=1):
        a_values = np.array(step['outputs']['a_values'], dtype=float)
        energies = np.array(step['outputs']['energies'], dtype=float)
        a_interp = float(step['outputs'].get('a_interp', np.nan))
        a_opt = float(step['outputs'].get('a_opt', np.nan))

        plt.figure(figsize=(8, 5))
        plt.plot(a_values, energies, 'o-')
        if not np.isnan(a_interp):
            plt.axvline(a_interp, color='orange', linestyle='--')
        if not np.isnan(a_opt):
            plt.axvline(a_opt, color='red', linestyle='-.')

        plt.title(f'Lattice constant scan #{idx}')
        plt.xlabel('Lattice constant a (Å)')
        plt.ylabel('Total energy (eV)')
        plt.show()
